Transaction costs. `05_backtest.ipynb` flags that costs aren't modelled. There's no real bid/ask data here for variance swaps or VIX futures, so the tradable quantity is proxied as the strategy's effective (leverage-scaled) exposure, position times the vol-target scalar, and charged a flat cost in basis points on its day-over-day change.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.vrp_strategy.backtest import raw_strategy_returns, vol_target_scale
from src.vrp_strategy.metrics import performance_metrics, print_metrics

DATA_PROCESSED = ROOT / "data" / "processed"
data = pd.read_csv(DATA_PROCESSED / "positions.csv", index_col=0, parse_dates=True)
SPLIT = "2016-01-01"
data = data[data.index >= SPLIT]

TARGET_VOL, VOL_WINDOW, SCALAR_CAP = 0.10, 126, 100.0

raw_ret = raw_strategy_returns(data["position"], data["iv_daily"], data["rv_daily"])
scaled_ret, scalar = vol_target_scale(raw_ret, TARGET_VOL, VOL_WINDOW, SCALAR_CAP)

# Exposure actually held through day t is position(t-1) scaled by that day's
# leverage, matching what generates strat_ret in raw_strategy_returns / vol_target_scale.
data["effective_position"] = data["position"].shift(1) * scalar
data["turnover"] = data["effective_position"].diff().abs()
data["gross_ret"] = scaled_ret
data = data.dropna(subset=["gross_ret", "turnover"])

print(f"Rows: {len(data)}")
print(f"Mean daily turnover:  {data['turnover'].mean():.2f}x notional")
print(f"Annualised turnover:  {data['turnover'].sum() / (len(data) / 252):.1f}x notional / year")

Rows: 2484
Mean daily turnover:  5.24x notional
Annualised turnover:  1319.3x notional / year


In [2]:
# Split turnover into (a) re-timing the VRP signal itself (0-1 range) and
# (b) the daily vol-target releveraging (the scalar, up to 100x), to see
# which one is actually driving the cost.
signal_turnover = data["position"].diff().abs()
scalar_turnover = scalar.reindex(data.index).diff().abs()

ann_signal_turnover = signal_turnover.sum() / (len(data) / 252)
ann_scalar_turnover = scalar_turnover.sum() / (len(data) / 252)

print(f"Annualised turnover, VRP signal only (0-1 range):   {ann_signal_turnover:6.1f}x/yr")
print(f"Annualised turnover, vol-target scalar (leverage):  {ann_scalar_turnover:6.1f}x/yr")

for bps in [1, 3, 5]:
    net_signal_only = data["gross_ret"] - (bps / 10_000) * signal_turnover
    m = performance_metrics(net_signal_only.dropna())
    print(f"  {bps}bps on signal-only turnover -> "
          f"ann. return {m['ann_return']*100:6.2f}%   Sharpe {m['sharpe']:.3f}")

print("\nThe VRP signal itself barely trades - almost all turnover comes from the")
print("vol-target scalar re-levering daily. This is a leverage-cycling cost, not a")
print("signal-timing cost, which matters for reading the sensitivity table below.")

Annualised turnover, VRP signal only (0-1 range):     15.2x/yr
Annualised turnover, vol-target scalar (leverage):    70.9x/yr
  1bps on signal-only turnover -> ann. return  31.79%   Sharpe 3.064
  3bps on signal-only turnover -> ann. return  31.49%   Sharpe 3.032
  5bps on signal-only turnover -> ann. return  31.18%   Sharpe 3.000

The VRP signal itself barely trades - almost all turnover comes from the
vol-target scalar re-levering daily. This is a leverage-cycling cost, not a
signal-timing cost, which matters for reading the sensitivity table below.


Cost sensitivity: 1 / 3 / 5 / 10 bps on total (signal + releveraging) turnover.

In [3]:
COST_LEVELS_BPS = [0, 1, 3, 5, 10]

rows = {}
for bps in COST_LEVELS_BPS:
    cost = (bps / 10_000) * data["turnover"]
    net_ret = data["gross_ret"] - cost
    rows[f"{bps}bps"] = performance_metrics(net_ret)

sensitivity = pd.DataFrame(rows).T[
    ["ann_return", "ann_vol", "sharpe", "sortino", "calmar", "max_drawdown", "win_rate"]
]
print(sensitivity)
sensitivity.to_csv(DATA_PROCESSED / "transaction_costs.csv")
print("\nSaved transaction_costs.csv")

       ann_return   ann_vol    sharpe   sortino    calmar  max_drawdown  \
0bps     0.319430  0.103714  3.079900  1.479031  1.626745     -0.196361   
1bps     0.187503  0.103886  1.804882  1.026103  0.694853     -0.269846   
3bps    -0.076351  0.108641 -0.702784 -0.513532 -0.109431     -0.697705   
5bps    -0.340204  0.118596 -2.868608 -2.356688 -0.351200     -0.968692   
10bps   -0.999838  0.158736 -6.298765 -5.688127 -0.999884     -0.999955   

       win_rate  
0bps   0.760870  
1bps   0.690419  
3bps   0.528583  
5bps   0.415459  
10bps  0.275765  

Saved transaction_costs.csv


In [4]:
DEFAULT_BPS = 3
net_ret_default = data["gross_ret"] - (DEFAULT_BPS / 10_000) * data["turnover"]

gross_m = performance_metrics(data["gross_ret"])
net_m = performance_metrics(net_ret_default)
print_metrics(gross_m, "Gross (no costs)")
print_metrics(net_m, f"Net of {DEFAULT_BPS}bps on total turnover")

print(f"\nAt {DEFAULT_BPS}bps on total turnover, annualised return moves from "
      f"{gross_m['ann_return']*100:.2f}% to {net_m['ann_return']*100:.2f}% "
      f"({(net_m['ann_return']-gross_m['ann_return'])*100:+.2f}pp/yr)")


Gross (no costs)
  Annualised return:  31.94%
  Annualised vol:     10.37%
  Sharpe ratio:       3.080
  Sortino ratio:      1.479
  Calmar ratio:       1.627
  Max drawdown:       -19.64%
  Win rate:           76.1%

Net of 3bps on total turnover
  Annualised return:  -7.64%
  Annualised vol:     10.86%
  Sharpe ratio:       -0.703
  Sortino ratio:      -0.514
  Calmar ratio:       -0.109
  Max drawdown:       -69.77%
  Win rate:           52.9%

At 3bps on total turnover, annualised return moves from 31.94% to -7.64% (-39.58pp/yr)


In [5]:
print("Caveat 1: charges cost on every daily re-leveraging move, i.e. assumes the")
print("          desk mechanically re-hedges to the vol target every single day. A")
print("          damped update rule (rebalance bands, a slower vol estimate) would")
print("          cut turnover and cost without giving up much vol-targeting benefit.")
print("Caveat 2: only charges a trading cost on rebalancing, not the financing cost")
print("          of carrying leverage between rebalances.")
print("Caveat 3: bps is a flat placeholder. Real variance-swap/VIX-futures spreads")
print("          widen in stress, exactly when turnover also spikes, so a regime-")
print("          scaled cost would likely show a bigger drag than shown above.")

Caveat 1: charges cost on every daily re-leveraging move, i.e. assumes the
          desk mechanically re-hedges to the vol target every single day. A
          damped update rule (rebalance bands, a slower vol estimate) would
          cut turnover and cost without giving up much vol-targeting benefit.
Caveat 2: only charges a trading cost on rebalancing, not the financing cost
          of carrying leverage between rebalances.
Caveat 3: bps is a flat placeholder. Real variance-swap/VIX-futures spreads
          widen in stress, exactly when turnover also spikes, so a regime-
          scaled cost would likely show a bigger drag than shown above.


In [6]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
fig.suptitle("Transaction Cost Impact", fontsize=14, fontweight="bold")

gross_cum = (1 + data["gross_ret"]).cumprod()
net_cum = (1 + net_ret_default).cumprod()
axes[0].plot(gross_cum.index, gross_cum.values, color="#2563eb", linewidth=1.2, label="Gross")
axes[0].plot(net_cum.index, net_cum.values, color="#dc2626", linewidth=1.2, label=f"Net ({DEFAULT_BPS}bps)")
axes[0].set_ylabel("Cumulative return")
axes[0].set_title("Cumulative performance — gross vs net of costs")
axes[0].legend(loc="upper left", fontsize=9)

gross_dd = gross_cum / gross_cum.cummax() - 1
net_dd = net_cum / net_cum.cummax() - 1
axes[1].plot(gross_dd.index, gross_dd.values * 100, color="#2563eb", linewidth=1.0, label="Gross")
axes[1].plot(net_dd.index, net_dd.values * 100, color="#dc2626", linewidth=1.0, label=f"Net ({DEFAULT_BPS}bps)")
axes[1].set_ylabel("Drawdown (%)")
axes[1].set_title("Drawdown — gross vs net of costs")
axes[1].legend(loc="lower left", fontsize=9)

axes[2].fill_between(data.index, data["turnover"], color="#7c3aed", alpha=0.5)
axes[2].set_ylabel("Turnover (notional x/day)")
axes[2].set_title("Daily turnover in effective (leverage-scaled) exposure")

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(axis="x", linestyle="--", linewidth=0.4, alpha=0.5)

plt.tight_layout()
plt.savefig(DATA_PROCESSED / "transaction_costs.png", dpi=150, bbox_inches="tight")
plt.show()